# Gemma 4 Code Representation Pilot (Kaggle Environment)

### Kaggle Session Configuration Checklist:
1. **Accelerator:** Under the right panel (Session options/Settings), select **GPU T4 x2** (or **GPU P100**).
2. **Internet:** Toggle to **On** (required to download Hugging Face model weights).
3. **Hugging Face Token:** If `google/gemma-4-E4B-it` requires gated access, enter your token in Cell 1 or via `Add-ons` -> `Secrets` (key: `HF_TOKEN`).
4. **Project Files:** Upload `gemma_paper_handoff.zip` using the **+ Add Input** button (or drag-and-drop into `/kaggle/working`).


In [ ]:
# === CELL 1: SETUP & ENVIRONMENT RESTORATION ===
import os, sys, zipfile, shutil

# 1. Hugging Face Access (Optional if unauthenticated, or set secret HF_TOKEN)
# os.environ["HF_TOKEN"] = "hf_..."  # Uncomment and insert if not using Kaggle Secrets

# 2. Install required quantization and model dependencies
!pip install -q "bitsandbytes>=0.50.0" accelerate datasets

# 3. Auto-detect working directory
BASE_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "/content"
if BASE_DIR not in sys.path:
    sys.path.insert(0, BASE_DIR)

# 4. Search and unpack gemma_paper_handoff.zip
candidate_zips = [
    f"{BASE_DIR}/gemma_paper_handoff.zip",
    "/kaggle/input/gemma-paper-handoff/gemma_paper_handoff.zip",
    "/kaggle/input/gemma_paper_handoff.zip"
]
unpacked = False
for p in candidate_zips:
    if os.path.exists(p):
        print(f"Unpacking {p} into {BASE_DIR}...")
        with zipfile.ZipFile(p, "r") as z:
            z.extractall(BASE_DIR)
        unpacked = True
        break

if not unpacked:
    # Check if files already present in working directory
    if os.path.exists(os.path.join(BASE_DIR, "gemma_paper")):
        print(f"Project files already present in {BASE_DIR}.")
    else:
        print("[!] Please upload gemma_paper_handoff.zip into /kaggle/working or add it as an input dataset.")

# 5. Check CUDA Availability
import torch
assert torch.cuda.is_available(), "No GPU detected! In Kaggle right sidebar, set Accelerator to GPU T4 x2 or P100."
print(f"CUDA Acceleration Active : {torch.cuda.is_available()}")
print(f"Primary GPU Device       : {torch.cuda.get_device_name(0)}")
print(f"GPU Total VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


In [ ]:
# === CELL 2: PREPARE REPOSITORY AT PINNED BASE COMMIT ===
from gemma_paper.dataset_prep import prepare_repository_at_base_commit

TASK_ID = "marshmallow-code__marshmallow-1343"
REPO_NAME = "marshmallow-code/marshmallow"
BASE_COMMIT = "2be2d83a1a9a6d3d9b85804f3ab545cecc409bb0"
REPO_DEST = os.path.join(BASE_DIR, "task_repos", TASK_ID.replace("/", "__"))

print(f"Preparing repository at base commit: {BASE_COMMIT}...")
prepare_repository_at_base_commit(REPO_NAME, BASE_COMMIT, REPO_DEST)
print(f"Repository ready at: {REPO_DEST}")


In [ ]:
# === CELL 3: LOAD GEMMA 4 IN 8-BIT INT8 & AUDIT PLACEMENT ===
from gemma_paper.model_loader import load_gemma4_and_processor, verify_quantization_and_placement

MODEL_ID = "google/gemma-4-E4B-it"
MODEL_REVISION = "4ecc2ba63bcac23ba4a767fc87025b1450f84c9c"

# Loads real model in 8-bit using BitsAndBytesConfig(load_in_8bit=True)
model, processor, unquantized_dtype = load_gemma4_and_processor(
    model_id=MODEL_ID,
    revision=MODEL_REVISION,
    use_4bit=False
)

# Reports Linear8bitLt layers and VRAM usage
quant_report = verify_quantization_and_placement(model)


In [ ]:
# === CELL 4: EXECUTE REAL MARSHMALLOW W INFERENCE ATTEMPT ===
import json
from gemma_paper.repair_runner import run_single_attempt

CONDITION = "W"
OUTPUT_DIR = os.path.join(BASE_DIR, "inference_output")
PACKET_FILE = os.path.join(BASE_DIR, "evidence_packets", f"{TASK_ID}__{CONDITION}.json")

with open(PACKET_FILE, "r", encoding="utf-8") as f:
    evidence_packet = json.load(f)

print(f"Executing real GPU generation on {TASK_ID} [Condition {CONDITION}]...")
record = run_single_attempt(
    model=model,
    processor=processor,
    task_id=TASK_ID,
    condition=CONDITION,
    evidence_packet=evidence_packet,
    output_dir=OUTPUT_DIR,
    repos_base_dir=os.path.join(BASE_DIR, "task_repos"),
    attempt=1
)

print("\n" + "=" * 80)
print("EXECUTION RESULTS (COPY AND PASTE INTO CHAT)")
print("=" * 80)
print(f"Checkpoint Key      : {record['checkpoint_key']}")
print(f"Execution Mode      : {record['execution_mode']}")
print(f"Input Tensor Tokens : {record['token_counts']['input_tokens']}")
print(f"Generated Tokens    : {record['token_counts']['generated_tokens']}")
print(f"Elapsed Time        : {record['elapsed_seconds']}s")
print(f"Peak VRAM Usage     : {record['memory_measurements']['after'].get('peak_mb', 'N/A')} MB")
print(f"Generation Status   : {record['generation_status']} (finish: {record['finish_reason']})")
print(f"Conversion Status   : {record['conversion_status']}")
print(f"Patch Applicability : {record['patch_applicability']}")
print(f"Test Resolution     : {record['test_resolution']}")
print(f"Prediction File     : {OUTPUT_DIR}/predictions/{record['checkpoint_key']}.json")
print("\n--- Extracted Model Patch ---")
print(record["model_patch"] if record["model_patch"] else "[No unified diff extracted]")
print("=" * 80)
